In [ ]:
import vis
from importlib import reload; reload(vis)
import os

path = "./dtlz2_d10_o5_p2000_g10000.json"

if os.path.exists(path):
    self = vis.Visualizer(from_path="./dtlz2_d10_o5_p2000_g10000.json")
else:    
    self = vis.Visualizer(
        from_problem="dtlz2",
        n_var=10, n_obj=5,
        pop_size=2000,
        n_offsprings=10,
        n_gen=10000
    )
    
    self.to_file(path)

In [ ]:
self.joint_xy

In [ ]:
import pandas as pd
import matplotlib as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from sklearn.cluster import DBSCAN, HDBSCAN

In [ ]:
kwargs = dict(
        threshold=0.3,
        clu = HDBSCAN(
            cluster_selection_epsilon=1.,
            min_cluster_size=10
        ),
        drop_intermediate=False
    )
    
self.get_overlapping_clusters(**kwargs)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
from scipy.spatial import ConvexHull
def get_convex_hull(points):
    return points.iloc[ConvexHull(points.values).vertices]

In [ ]:


def draw_clusters_in_dash(clusters, points):
    
    # kwargs = dict(
    #     threshold=th,
    #     clu = HDBSCAN(
    #         cluster_selection_epsilon=1.,
    #         min_cluster_size=10
    #     ),
    #     drop_intermediate=False
    # )
    
    # clusters = visualizer.get_overlapping_clusters(**kwargs)
    # points = visualizer.joint_xy

    fig = go.Figure()
    init = points.loc[clusters.index]
    fig.add_trace(go.Scatter(x=init[0], y=init[1], mode='markers', marker=dict(color='lightgray', size=5), showlegend=False))
    for i, c in enumerate(clusters):
        if c in sorted(clusters.columns):
            ec = pc.hex_to_rgb(px.colors.qualitative.D3[i]) + (1.0,)
    
            hulls = [
                get_convex_hull(dfk)
                for k, dfk in points.groupby(clusters[c])
                if k != -1
            ]
            for j in range(len(hulls)):
                x_coords = hulls[j][0].to_list()
                y_coords = hulls[j][1].to_list()
                fig.add_trace(go.Scatter(x=x_coords+[x_coords[0]], y=y_coords+[y_coords[0]], mode="lines", fill="toself", fillcolor=f"rgba{ec[:3] + (0.2,)}", line=dict(color=f"rgba{ec}"), showlegend=False))
    
            mask = clusters[c] != -1
            multi_cluster_mask = (clusters[clusters.columns] != -1).sum(axis=1) > 1
            
            # excluding points with multiple clusters
            remaining_df = points.loc[clusters.index][mask & ~multi_cluster_mask]
            fig.add_trace(go.Scatter(x=remaining_df[0], y=remaining_df[1], mode='markers', marker=dict(color=f'rgba{ec}', size=5), name=c))
    
            # points with multiple clusters
            multi_cluster_df = points.loc[clusters.index][multi_cluster_mask]
            fig.add_trace(go.Scatter(x=multi_cluster_df[0], y=multi_cluster_df[1], mode='markers', marker=dict(color='black', size=5), name='multiple_cluster', showlegend=False))
    
    fig.update_layout(height=800, template="plotly_white")
    return fig

Interactions:
- Allow user to select a specific specialization cluster
- Show a distribution of the decision values (refer back to the box plot dist in Dustin's notebook])

In [ ]:
selected = ['f1', 'f0']
clusters = self.get_overlapping_clusters(**kwargs)[sorted(selected)]
points = self.joint_xy
draw_clusters_in_dash(clusters, points)

In [ ]:
def draw_obj_dec_dist(df, clusters, selected):
    dvars = [c for c in self.df.columns if 'x' in c]
    data = pd.concat([
        df.loc[clusters.index[clusters[c] == i], dvars]\
            .assign(y=f'{c}-{i}', ovar=c)
        for c in clusters
        for i in clusters[c].unique()
        if i != -1
    ])
    
    tmp = pd.melt(data[data.ovar.isin(selected)], id_vars=['y', 'ovar'], value_vars=dvars, var_name='dvar').sort_values(['y', 'dvar', 'ovar'])

    fig = px.box(tmp, x='value', y='y', facet_col='dvar', color='ovar', color_discrete_sequence=px.colors.qualitative.D3)
    fig.update_traces(width=0.5)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_layout(yaxis_title='Specialization (Cluster)', template="plotly_white")
    fig.update_xaxes(title_text = '')
    return fig

In [ ]:
draw_obj_dec_dist(self.df, clusters, selected)

In [ ]:
from dash import Dash, html, dcc, Output, Input, State, no_update
import dash_bootstrap_components as dbc

app = Dash(__name__, suppress_callback_exceptions=True, external_stylesheets=[dbc.themes.BOOTSTRAP])

clusters = self.get_overlapping_clusters(**kwargs)

app.layout = html.Div([
    html.Div([
        html.Div([
            html.P('Specialization (Cluster)', style={'marginBottom': '5px', 'padding': '1%'}),
            dcc.Dropdown(id='cluster-dropdown', placeholder='Select a cluster', options=clusters.columns, value=sorted(clusters.columns), multi=True, style={'width': '40vw'})
        ], style={'border': '1px solid #d3d3d3', 'marginRight': '1%'}),
        html.Div([
            html.P('Threshold (Epsilon)', style={'marginLeft': '3%', 'marginBottom': '5px'}), 
            dcc.Slider(0, 0.5, 0.1, value=0.3, id='th-slider')
        ], style={'width': '30vw', 'border': '1px solid #d3d3d3'}),
    ], style={'display': 'flex', 'padding': '5px'}),
    dcc.Loading(
        id='loading-1',
        children=html.Div(id='graph-container', children=[
            dcc.Graph(id='cluster-scatterplot', style={'width': '50%'}),
            dcc.Graph(id='obj-dec-boxplot', style={'width': '50%'})
        ], style={'display': 'none'})
    )
    
])

@app.callback(
    Output('cluster-scatterplot', 'figure'),
    Output('graph-container', 'style'),
    Output('obj-dec-boxplot', 'figure'),
    Input('cluster-dropdown', 'value'),
    Input('th-slider', 'value')
)

def draw_scatter(selected_clusters, th):
    if len(selected_clusters) > 0:
            
        kwargs = dict(
            threshold=th,
            clu = HDBSCAN(
                cluster_selection_epsilon=1.,
                min_cluster_size=10
            ),
            drop_intermediate=False
        )

        clusters = self.get_overlapping_clusters(**kwargs)[sorted(selected_clusters)]
        points = self.joint_xy
        return draw_clusters_in_dash(clusters, points), {'display': 'flex'}, draw_obj_dec_dist(self.df, clusters, selected_clusters)
    else:
        return no_update, {'display': 'none'}, no_update

app.run_server(port=8400, debug=True, jupyter_mode='external')